# Tutorial A02: S1-GRiTS Full Pipeline Walkthrough

This notebook is the primary guide for the `s1grits` project. It covers the complete workflow from environment setup to data visualization, all runnable inside Jupyter.

---

## Overview

This tutorial covers:

1. Environment and installation check
2. ASF authentication (`.netrc` setup)
3. Configuration (YAML config file)
4. Running the pipeline (`s1grits process`)
5. Catalog management (`s1grits catalog rebuild/validate`)
6. COG validation (`s1grits cog validate`)
7. Zarr inspection (`s1grits zarr inspect`, `zarr fix-time`)
8. Time series plot (`s1grits timeseries plot`)
9. Export PNG (`s1grits export png`)
10. Mosaic creation (`s1grits mosaic create`)


---

## 1. Environment and Installation Check

### 1.1 Recommended Setup (Conda + libmamba)

Run the following commands in a terminal:

```bash
# Navigate to the project root
cd ~/S1-GRiTS

# Enable the libmamba solver for faster, more reliable dependency resolution
conda install -n base conda-libmamba-solver

# Create the environment from environment.yml
conda env create -f environment.yml --solver=libmamba

# Activate the environment
conda activate py312_s1grits

# Install the project in editable mode
pip install -e .

# Register the kernel for Jupyter (optional)
python -m ipykernel install --user --name py312_s1grits --display-name "Python (py312_s1grits)"

# Launch Jupyter
jupyter notebook
```


In [1]:
# Verify that s1grits is installed in the active environment
import subprocess
import sys

print("sys.executable:", sys.executable)
print("python version :", sys.version)

p = subprocess.run("where s1grits", shell=True, text=True, capture_output=True)
print("returncode:", p.returncode)
print("stdout:\n", p.stdout)
print("stderr:\n", p.stderr)


sys.executable: C:\Users\raokeyi\anaconda3\envs\py312_s1grits_v100\python.exe
python version : 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:36:12) [MSC v.1944 64 bit (AMD64)]
returncode: 0
stdout:
 C:\Users\raokeyi\anaconda3\envs\py312_s1grits_v100\Scripts\s1grits.exe

stderr:
 


---

## 2. ASF Authentication (.netrc Setup)

`s1grits` uses `asf_search` to access OPERA RTC-S1 products. All downloads require a NASA Earthdata account with ASF DAAC authorization.

**Step 1**: Register at https://urs.earthdata.nasa.gov and log in to https://search.asf.alaska.edu to authorize ASF access.

**Step 2**: Go to https://urs.earthdata.nasa.gov/profile, click **Applications**, search for **Alaska Satellite Facility Data Access**, and accept the EULA.

**Step 3**: Create `~/.netrc` (Unix/macOS) or `%USERPROFILE%\.netrc` (Windows):

```
machine urs.earthdata.nasa.gov
  login YOUR_USERNAME
  password YOUR_PASSWORD
```

**Step 4**: Enable `.netrc` in Python:

```python
import asf_search as asf
session = asf.ASFSession()
session.trust_env = True  # Allow reading .netrc and proxy environment config
```


---

## 3. Configuration (YAML Config File)

All processing parameters are defined in a single YAML configuration file: `config/s1grits_config_base_en.yaml`.

This file controls:

- **Input**: MGRS tiles or WKT ROI, time range, data source
- **Processing**: Orbit direction, monthly compositing strategy, speckle correction, band generation
- **Output**: COG path, Zarr path, catalog location
- **Runtime**: Parallelism, memory limits, retry behavior, log level

For first-time use, you only need to set two things:

1. **Spatial extent** (`roi.wkt`): A WKT polygon (copy from ASF Vertex)
2. **Time range** (`time_range`): Either a list of years or a full-range mode

**Example — specific years:**

```yaml
time_range:
  mode: "years"
  years: [2024, 2025]
```

**Example — full archive:**

```yaml
time_range:
  mode: "full"
  full_mode:
    end_year: 2025
```


In [2]:
# Load and inspect the processing config
from pathlib import Path
import yaml

def read_yaml(p: Path):
    with p.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)

# Locate config relative to the project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent
proc_yml = PROJECT_ROOT / "config" / "s1grits_config_base_en.yaml"
print("Config path:", proc_yml)
print("Exists:", proc_yml.exists())

if proc_yml.exists():
    proc_cfg = read_yaml(proc_yml)
    print("\n--- Top-level config keys ---")
    print(list(proc_cfg.keys()))
else:
    raise FileNotFoundError(f"Config not found: {proc_yml}")

Config path: D:\Project\claude-demo\S1-GRiTS-V100\config\s1grits_config_base_en.yaml
Exists: True

--- Top-level config keys ---
['roi', 'time', 'output', 'parallel', 'memory', 'processing', 'logging']


---

## 4. Running the Pipeline (CLIRunner)

`CLIRunner` is a Jupyter-friendly adapter for the `s1grits` CLI. It launches CLI commands via `subprocess`, captures stdout/stderr line by line, and displays output in a notebook-friendly format.

```
+---------------+
| s1grits CLI   |  <- Core logic, stable, reproducible
+-------^-------+
        | subprocess
+-------+-------+
| CLIRunner     |  <- Notebook / Windows / encoding adapter
+-------^-------+
        | Python API
+-------+-------+
| Jupyter       |  <- Interactive visualization and review
+---------------+
```


In [3]:
from s1grits.notebook_utils import CLIRunner
import os

# Force unbuffered output and color support
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["FORCE_COLOR"] = "1"

# Initialize the runner pointing at the project root
runner = CLIRunner(
    project_root=PROJECT_ROOT
    # Optional overrides:
    # max_display_lines=30,
    # refresh_rate=0.3,
    # enable_filter=False,
    # filter_keywords=["ERROR", "WARNING", "Tile", "Processing"],
)

### 4.1 Run the Processing Pipeline


In [4]:
# Run the full s1grits processing pipeline
# Equivalent to: s1grits process_scenes --config config/s1grits_config_base_en.yaml
cfg = runner.get_config_path("s1grits_config_base_en.yaml")
process_result = runner.run(["s1grits", "process_scenes", "--config", str(cfg)], stream=True, check=True)

Batches: 100%|██████████| 6/6 [00:19<00:00,  3.30s/it]
      Batch 1/1 done — wrote 1 month(s): 2026-02
  [4/4] Done: 1 month(s) written
───────────────────────── Processing Results Summary ──────────────────────────
│ Total MGRS Tiles │      1 │
│ Success          │      1 │
│ Failed           │      0 │
│ Success Rate     │ 100.0% │
│ MGRS Tile    │ Status │  Months │       Size │ Path/Error                   │
Success!


---

## 5. Catalog Management

S1-GRiTS uses a two-level catalog system to index all processed products:

**Tile-level catalog** (one per MGRS tile + orbit direction):
```
output/{TILE_ID}_{DIRECTION}/catalog.parquet
```

**Global catalog** (aggregates all tile catalogs):
```
output/catalog.parquet
```

Output directory structure:

```
output/
├── catalog.parquet                    # Global metadata index
├── 50RKV_ASCENDING/
│   ├── cog/                           # Cloud-Optimized GeoTIFF files
│   ├── zarr/S1_monthly.zarr/          # Zarr time-series store
│   │   ├── VV_dB/  VH_dB/  Ratio/  RVI/
│   │   └── time/  x/  y/
│   ├── preview/                       # 300m preview images
│   └── catalog.parquet
└── 50RKV_DESCENDING/
    └── ...
```


In [18]:
# Rebuild the global catalog (useful after manual data transfers or partial runs)
catalog_rebuild = runner.run("s1grits catalog rebuild --output-dir ./output_hARDCp")

# Validate the catalog
catalog_validate = runner.run("s1grits catalog validate --output-dir ./output_hARDCp")

# Inspect the catalog
catalog_validate = runner.run("s1grits catalog inspect --output-dir ./output_hARDCp")

─────────────────────────── Rebuild Catalog + STAC ────────────────────────────
Output directory: ./output_hARDCp

INFO   Catalog rebuilt successfully
       Tiles:   1
       Records: 1
       Catalog: output_hARDCp\catalog.parquet
────────────────────────────── Validate Catalog ───────────────────────────────
Catalog: output_hARDCp\catalog.parquet

INFO   Catalog schema is valid
       Records: 1
────────────────────────────── Catalog Coverage ───────────────────────────────
Output directory: ./output_hARDCp

Total records:  1
MGRS tiles:     1
Date range:     2026-02-01 to 2026-02-01
Total months:   1
Directions:     ASCENDING
                                                                               
                               Coverage by Tile                                
┌──────────┬────────────┬───────┬─────────┬───────┬─────────┬─────────────────┐
│ Tile     │ Direction  │ Mont… │ Expect… │ Miss… │ Comple… │ Range           │
├──────────┼────────────┼───────┼─────────

In [20]:
# Show missing months and COG file status for all directions of a tile
catalog_validate = runner.run("s1grits tile inspect --tile 50RKV --output-dir ./output_hARDCp")


──────────────────────────────────────────────────────────── Tile: 50RKV ────────────────────────────────────────────────────────────

ASCENDING
  Present months:  1
  Expected months: 1
  Date range:      2026-02 ~ 2026-02
  Completeness:    100.0%

  No missing months -- complete time series
────────────────────────────────────────────────────────────


---

## Summary: All CLI Commands

| Command | Description |
| :--- | :--- |
| `s1grits process_scenes --config config.yaml` | Run the full download and processing pipeline |
| `s1grits catalog rebuild --output-dir ./output` | Rebuild the global catalog from tile-level catalogs |
| `s1grits catalog validate --output-dir ./output` | Validate catalog integrity |
| `s1grits cog validate --output-dir ./output --sample 10` | Validate COG files (random sample) |
| `s1grits zarr inspect --tile TILE --direction DIR` | Inspect a Zarr dataset structure |
| `s1grits zarr fix-time --tile TILE --direction DIR` | Fix time dimension ordering in Zarr |
| `s1grits timeseries plot --tile TILE --direction DIR --pixel R C` | Plot per-pixel time series |
| `s1grits export png --tile TILE --direction DIR --month YYYY-MM` | Export a false-color composite PNG |
| `s1grits mosaic create --month YYYY-MM --direction DIR --mgrs-prefix XX` | Create a multi-tile VRT mosaic |
| `s1grits report coverage --output-dir ./output` | Generate a data coverage report |
